In [1]:
!pip install tensorflow

import zipfile
import re
import numpy as np
import pandas as pd
import tensorflow as tf

from google.colab import files
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [3]:
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall()

print("ZIP File Extracted Successfully")

Saving IMDB.zip to IMDB (2).zip
ZIP File Extracted Successfully


In [4]:
df = pd.read_csv("IMDB Dataset.csv")

print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Dataset Shape:
(50000, 2)

Missing Values:
review       0
sentiment    0
dtype: int64


In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z ]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

df['review'] = df['review'].apply(clean_text)

print(df.head())

                                              review sentiment
0  one of the other reviewers has mentioned that ...  positive
1  a wonderful little production the filming tech...  positive
2  i thought this was a wonderful way to spend ti...  positive
3  basically theres a family where a little boy j...  negative
4  petter matteis love in the time of money is a ...  positive


In [6]:
df['sentiment'] = df['sentiment'].map({
    'positive':1,
    'negative':0
})

print(df.head())

                                              review  sentiment
0  one of the other reviewers has mentioned that ...          1
1  a wonderful little production the filming tech...          1
2  i thought this was a wonderful way to spend ti...          1
3  basically theres a family where a little boy j...          0
4  petter matteis love in the time of money is a ...          1


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.2,
    random_state=42
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 40000
Testing Samples: 10000


In [8]:
VOCAB_SIZE = 10000
MAX_LENGTH = 200

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

X_train = tokenizer.texts_to_sequences(X_train)
X_test = tokenizer.texts_to_sequences(X_test)

X_train = pad_sequences(
    X_train,
    maxlen=MAX_LENGTH,
    padding='post',
    truncating='post'
)

X_test = pad_sequences(
    X_test,
    maxlen=MAX_LENGTH,
    padding='post',
    truncating='post'
)

print(X_train.shape)
print(X_test.shape)

(40000, 200)
(10000, 200)


In [9]:
model = Sequential()

model.add(Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=128,
    input_length=MAX_LENGTH
))

model.add(LSTM(
    128,
    dropout=0.2,
    recurrent_dropout=0.2
))

model.add(Dense(64, activation='relu'))

model.add(Dropout(0.5))

model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 302s 595ms/step - accuracy: 0.5321 - loss: 0.6901 - val_accuracy: 0.5669 - val_loss: 0.6783
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 322s 596ms/step - accuracy: 0.6027 - loss: 0.6595 - val_accuracy: 0.7424 - val_loss: 0.5611
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 300s 599ms/step - accuracy: 0.7380 - loss: 0.5513 - val_accuracy: 0.8314 - val_loss: 0.3961
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 321s 597ms/step - accuracy: 0.8723 - loss: 0.3285 - val_accuracy: 0.8624 - val_loss: 0.3327
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 294s 587ms/step - accuracy: 0.9121 - loss: 0.2447 - val_accuracy: 0.8586 - val_loss: 0.3593


In [11]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 24s 76ms/step - accuracy: 0.8683 - loss: 0.3460
Test Loss: 0.345988929271698
Test Accuracy: 0.8683000206947327


In [14]:
def predict_sentiment(review):

    review = clean_text(review)

    sequence = tokenizer.texts_to_sequences([review])

    padded = pad_sequences(
        sequence,
        maxlen=MAX_LENGTH,
        padding='post',
        truncating='post'
    )

    prediction = model.predict(padded)

    if prediction[0][0] >= 0.5:
        print("Positive Review 😊")
        print("Confidence:", round(prediction[0][0]*100,2),"%")
    else:
        print("Negative Review 😞")
        print("Confidence:", round((1-prediction[0][0])*100,2),"%")

In [15]:
predict_sentiment("This movie was amazing. I loved it.")

predict_sentiment("Worst movie ever. Waste of time.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 931ms/step
Positive Review 😊
Confidence: 96.66 %
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
Negative Review 😞
Confidence: 93.28 %


In [16]:
model.save("imdb_lstm_model.h5")

print("Model Saved Successfully")

Model Saved Successfully
